# 03 — LightGBM con Features de Lag
**Entrega 3.** Un modelo LightGBM por índice con lag features, predicción autorregresiva.

In [1]:
import sys, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.insert(0, ".")
from utils import (load_data, compute_rmse, make_submission, train_val_split,
                   create_lag_features, add_calendar_features, add_log_returns,
                   find_ghost_source, INDEX_COLS)

data         = load_data()
train_full   = data["train_indices"][INDEX_COLS]
test_dates   = data["test_dates"].index
train, val   = train_val_split(train_full, val_size=252)
macro        = data.get("train_macro_factors")
network      = data.get("train_network_metrics")
macro_test   = data.get("test_macro_factors")
network_test = data.get("test_network_metrics")

LAGS    = (1, 2, 3, 5, 10, 20, 60)
WINDOWS = (5, 10, 20, 60)
print(f"Train: {train.shape}  |  Val: {val.shape}  |  Test: {len(test_dates)} días")


Train: (2095, 6)  |  Val: (252, 6)  |  Test: 262 días


## Ghost detection — ¿qué índice imita Index_D?

In [2]:
find_ghost_source(train_full, target_col='Index_D', max_lag=30)

Ghost source: Index_A at lag 3 (corr=0.9999)


('Index_A', 3, np.float64(0.9999246961949881))

## Feature engineering

In [3]:
def build_features(indices, macro=None, network=None):
    df = indices[INDEX_COLS].copy()
    df = add_log_returns(df)
    df = create_lag_features(df, lags=LAGS, windows=WINDOWS)
    df = add_calendar_features(df)
    if macro is not None:
        df = pd.concat([df, macro.reindex(df.index).ffill()], axis=1)
    if network is not None:
        df = pd.concat([df, network.reindex(df.index).ffill()], axis=1)
    return df

def prepare_xy(feats, targets):
    rename = {c: f"__t_{c}" for c in INDEX_COLS}
    combined = pd.concat([feats, targets.rename(columns=rename)], axis=1).dropna()
    tgt_cols = list(rename.values())
    return combined.drop(columns=tgt_cols).values, combined[tgt_cols].values, combined.index

macro_tr = macro.iloc[:-252] if macro is not None else None
net_tr   = network.iloc[:-252] if network is not None else None
feats_tr = build_features(train, macro_tr, net_tr)
X_tr, y_tr, _ = prepare_xy(feats_tr, train)
feature_names  = list(feats_tr.dropna().columns)
print(f"Feature matrix: {X_tr.shape}")


Feature matrix: (2034, 201)


## Entrenamiento

In [4]:
def train_lgbm(X, y, n_estimators=500):
    try:
        import lightgbm as lgb
        models = []
        for i, col in enumerate(INDEX_COLS):
            m = lgb.LGBMRegressor(n_estimators=n_estimators, learning_rate=0.05,
                                  num_leaves=63, subsample=0.8,
                                  colsample_bytree=0.8, verbose=-1)
            m.fit(X, y[:, i])
            models.append(m)
        return models, "lgbm"
    except ImportError:
        pass
    try:
        from xgboost import XGBRegressor
        models = [XGBRegressor(n_estimators=n_estimators, learning_rate=0.05, verbosity=0
                               ).fit(X, y[:, i]) for i in range(y.shape[1])]
        return models, "xgb"
    except ImportError:
        pass
    from sklearn.ensemble import GradientBoostingRegressor
    models = [GradientBoostingRegressor(n_estimators=200).fit(X, y[:, i])
              for i in range(y.shape[1])]
    return models, "sklearn_gbr"

models, lib = train_lgbm(X_tr, y_tr)
print(f"Modelos entrenados: {lib}")


Modelos entrenados: sklearn_gbr


## Predicción autorregresiva

In [5]:
def autoreg_predict(models, history, dates,
                    macro_all=None, net_all=None, feature_names=None):
    history = history[INDEX_COLS].copy()
    preds   = []
    for date in dates:
        feats = build_features(
            history,
            macro   = macro_all.loc[:date] if macro_all is not None else None,
            network = net_all.loc[:date]   if net_all   is not None else None,
        )
        row = feats.dropna().iloc[[-1]]
        if feature_names is not None:
            for c in feature_names:
                if c not in row.columns:
                    row[c] = 0.0
            row = row[feature_names]
        y_hat = np.array([m.predict(row.values)[0] for m in models])
        preds.append(y_hat)
        new_row = pd.DataFrame([y_hat], index=[date], columns=INDEX_COLS)
        history = pd.concat([history, new_row])
    return pd.DataFrame(preds, index=dates, columns=INDEX_COLS)


## Validación local

In [6]:
pred_val = autoreg_predict(models, train, val.index,
                           macro_all=pd.concat([macro]) if macro is not None else None,
                           net_all=pd.concat([network]) if network is not None else None,
                           feature_names=feature_names)
rmse = compute_rmse(val, pred_val)
print(f"[LightGBM] RMSE local = {rmse:,.2f}")
per = np.sqrt(((val.values - pred_val.values)**2).mean(axis=0))
for col, r in zip(INDEX_COLS, per):
    print(f"  {col}: {r:,.2f}")


[LightGBM] RMSE local = 3,279.20
  Index_A: 3,198.91
  Index_B: 395.32
  Index_C: 1.41
  Index_D: 3,329.14
  Index_E: 9.89
  Index_F: 12,740.50


## Generar submission

In [7]:
feats_full = build_features(train_full, macro, network)
X_full, y_full, _ = prepare_xy(feats_full, train_full)
fn_full = list(feats_full.dropna().columns)
models_full, _ = train_lgbm(X_full, y_full)

# Concatenar macro/network train+test para que el bucle autoregresivo
# pueda acceder a fechas de test en cada paso
import pandas as pd
macro_all_test   = pd.concat([macro, macro_test])   if macro   is not None else None
network_all_test = pd.concat([network, network_test]) if network is not None else None

pred_test = autoreg_predict(models_full, train_full, test_dates,
                            macro_all=macro_all_test, net_all=network_all_test,
                            feature_names=fn_full)
make_submission(pred_test, "submission_03_lgbm.csv")
pred_test.head()


Submission saved: c:\Users\1jose\Desktop\previa hackatlon\hackathon\submissions\submission_03_lgbm.csv


,Index_A,Index_B,Index_C,Index_D,Index_E,Index_F
Date,,,,,,
2024-01-01,16819.935562,4769.124260,20.257338,16825.757134,128.947234,41999.876098
2024-01-02,16828.379621,4769.251755,20.264250,16823.478859,128.909377,41947.202946
2024-01-03,16817.903553,4768.515257,20.259973,16822.839509,128.943675,41965.728385
2024-01-04,16817.022553,4768.916942,20.259930,16820.383099,128.932608,41843.847211
2024-01-05,16816.150868,4770.915007,20.258246,16826.154202,128.970031,41909.092898
